In [172]:
import pandas as pd
import re
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\Aryan
[nltk_data]     Chauhan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

->Preprocessing and Cleaning
->Train test split
->BOW,TF-IDF,Word2Vec
->Train ML Algos

In [173]:
df=pd.read_csv('all_kindle_review.csv')

In [174]:
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [175]:
df=df[['reviewText','rating']]

In [176]:
df['reviewText'].isnull().count()

np.int64(12000)

In [177]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [178]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [179]:
#positivereivewis1andnegativereviewis1
df['rating']=df['rating'].apply(lambda x :0 if x<3 else 1)

In [180]:
df['rating']

0        1
1        1
2        1
3        1
4        1
        ..
11995    1
11996    1
11997    1
11998    0
11999    1
Name: rating, Length: 12000, dtype: int64

In [181]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

Preprocessing

In [182]:
#Loweringallthecases
df['reviewText']=df['reviewText'].str.lower()

In [183]:
from bs4 import BeautifulSoup

In [184]:
stop_words_set = set(stopwords.words('english'))

HTML_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'https?://\S+|www\.\S+|ftp://\S+|ssh://\S+')
PUNCT_RE = re.compile(r'[^a-z A-Z 0-9\s-]')

def clean_text_fast(text):
    text = str(text)
    text = HTML_RE.sub('', text)
    text = URL_RE.sub('', text)
    
    text = PUNCT_RE.sub('', text)
    return " ".join([word for word in text.split() if word.lower() not in stop_words_set])

df['reviewText'] = df['reviewText'].apply(clean_text_fast)

In [185]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [186]:
from nltk import WordNetLemmatizer

In [187]:
Lemm=WordNetLemmatizer()

In [188]:
def lemmatize(text):
    return "".join([Lemm.lemmatize(word) for word in text.split()])

In [189]:
df['reviewtext']=df['reviewText'].apply(lambda x : lemmatize(x))

In [190]:
from sklearn.model_selection import train_test_split

In [191]:
X_train,X_test,Y_train,Y_test=train_test_split(df['reviewText'],df['rating'],test_size=0.20,random_state=42)

In [192]:
from sklearn.feature_extraction.text import CountVectorizer

In [193]:
bow=CountVectorizer()

In [194]:
X_train=bow.fit_transform(X_train).toarray()
X_test=bow.transform(X_test).toarray()

In [195]:
from sklearn.naive_bayes import GaussianNB

In [196]:
nb=GaussianNB()

In [197]:
model=nb.fit(X_train,Y_train)

In [198]:
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report

In [199]:
y_pred=model.predict(X_test)

In [204]:
CM=classification_report(Y_test,y_pred)
print(CM)

              precision    recall  f1-score   support

           0       0.41      0.60      0.49       803
           1       0.74      0.57      0.64      1597

    accuracy                           0.58      2400
   macro avg       0.57      0.58      0.56      2400
weighted avg       0.63      0.58      0.59      2400



In [206]:
acc=accuracy_score(Y_test,y_pred)
print(acc)

0.5766666666666667


In [209]:
cm1=confusion_matrix(Y_test,y_pred)
print(cm1)

[[481 322]
 [694 903]]
